In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import sys, os
sys.dont_write_bytecode = True
repo = os.path.abspath("..")
if repo not in sys.path:
    sys.path.insert(0, repo)

In [0]:
import pytest
assert pytest.main([f"{repo}/tests", "-q", "-p", "no:cacheprovider"]) == 0
from etl.migrate import run_migrations
run_migrations(spark, f"{repo}/sql/ddl", upto="300")

In [0]:
from etl.merge_fact import merge_all
merge_all(spark)

In [0]:
display(spark.sql("""
SELECT (SELECT count(*) FROM dld.core.fact_transaction) AS fact_rows,
       (SELECT count(*) FROM dld.core.fact_transaction_history) AS history_rows,
       (SELECT count(*) FROM dld.core.fact_transaction_history WHERE is_current) AS current_versions,
       (SELECT count(*) FROM (SELECT transaction_key FROM dld.core.fact_transaction_history
                              GROUP BY 1 HAVING count(*) > 1)) AS keys_with_history"""))

In [0]:
from etl.merge_fact import HASH_COLS
diffs = ", ".join(f"sum(CASE WHEN NOT (a.{c} <=> b.{c}) THEN 1 ELSE 0 END) AS {c}" for c in HASH_COLS)
display(spark.sql(f"""SELECT {diffs}
    FROM dld.core.fact_transaction_history a
    JOIN dld.core.fact_transaction_history b
      ON a.transaction_key = b.transaction_key AND NOT a.is_current AND b.is_current"""))

In [0]:
merge_all(spark)

In [0]:
for t in ["e1_residential_sale_index", "e2_residential_sale_index"]:
    print(t, [c for c in spark.table(f"dld.raw.{t}").columns if not c.startswith("_")])
display(spark.table("dld.raw.e2_residential_sale_index").orderBy("first_date_of_month").limit(5))

In [0]:
import pytest
assert pytest.main([f"{repo}/tests", "-q", "-p", "no:cacheprovider"]) == 0

spark.sql("DELETE FROM dld.core.fact_transaction")
spark.sql("DELETE FROM dld.core.fact_transaction_history")

from etl.merge_fact import merge_all
print("--- first load ---")
merge_all(spark)
print("--- re-run, must be all zeros ---")
merge_all(spark)

In [0]:
import pytest
assert pytest.main([f"{repo}/tests", "-q", "-p", "no:cacheprovider"]) == 0
from etl.migrate import run_migrations
run_migrations(spark, f"{repo}/sql/ddl", upto="302")

from checks.check_reconcile import run, series_sql
run(spark)

In [0]:
display(spark.sql(f"SELECT * FROM ({series_sql('quarterly')}) WHERE segment = 'all' ORDER BY period"))

In [0]:
from etl.migrate import file_hash

cols = [r.col_name for r in spark.sql("DESCRIBE dld.meta.reconcile_result").collect()]
print("grain already in table:", "grain" in cols)

if "grain" in cols:
    txt = open(f"{repo}/sql/ddl/302_reconcile_grain.sql").read()
    spark.sql("""INSERT INTO dld.meta.schema_migration
                 VALUES (:f, :h, 0, current_timestamp())""",
              args={"f": "302_reconcile_grain.sql", "h": file_hash(txt)})
    print("302 recorded as applied (baselined, 0 statements run)")

In [0]:
display(spark.sql("""
  SELECT a.property_usage AS e1_value, b.property_usage AS e2_value, count(*) AS rows
  FROM dld.core.fact_transaction_history a
  JOIN dld.core.fact_transaction_history b
    ON a.transaction_key = b.transaction_key AND NOT a.is_current AND b.is_current
  WHERE NOT (a.property_usage <=> b.property_usage)
  GROUP BY 1, 2 ORDER BY 3 DESC LIMIT 10"""))

display(spark.sql("""
  SELECT CASE WHEN a.project_id = -1 THEN 'no project' WHEN a.project_id = -2 THEN 'not matched' ELSE 'real id' END AS e1_project,
         CASE WHEN b.project_id = -1 THEN 'no project' WHEN b.project_id = -2 THEN 'not matched' ELSE 'real id' END AS e2_project,
         count(*) AS rows
  FROM dld.core.fact_transaction_history a
  JOIN dld.core.fact_transaction_history b
    ON a.transaction_key = b.transaction_key AND NOT a.is_current AND b.is_current
  WHERE NOT (a.project_id <=> b.project_id)
  GROUP BY 1, 2 ORDER BY 3 DESC"""))

In [0]:
import sys, os
sys.dont_write_bytecode = True
repo = os.path.abspath("..")
if repo not in sys.path:
    sys.path.insert(0, repo)
import pytest
assert pytest.main([f"{repo}/tests", "-q", "-p", "no:cacheprovider"]) == 0

from pyspark.sql import functions as F
from etl.conform import conform_all, TERMS
from etl.merge_fact import merge_all

conform_all(spark)                                   # staging rebuilt with the fixed term map

# any Arabic text left in English term columns?
stg = spark.table("dld.stg.transaction")
for t in TERMS:
    found = stg.where(F.col(t).rlike(r"\p{IsArabic}")).groupBy(t).count().collect()
    print(t.ljust(18), [(r[t], r["count"]) for r in found] or "none")

# our own vocabulary fix is not a source change: rebuild the history instead of recording fake versions
spark.sql("DELETE FROM dld.core.fact_transaction")
spark.sql("DELETE FROM dld.core.fact_transaction_history")
merge_all(spark)